# Time-Series Regression Case Study

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from IPython.display import display
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import acf

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

## 1. Load data and perform a minimal global audit

In [ ]:
path = "5_single_stock_feature_engineering_case.csv"
df_raw = pd.read_csv(path, parse_dates=["date"])

print("Raw shape:", df_raw.shape)
print("Date range:", df_raw["date"].min(), "to", df_raw["date"].max())
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("Missing values by column:")
display(df_raw.isna().sum().to_frame("missing_count").T)
print("\nDtypes:")
display(df_raw.dtypes.astype(str).to_frame("dtype").T)

In [ ]:
%matplotlib inline

summary_table = pd.DataFrame({
    "min": df_raw.drop(columns=["date"]).min(numeric_only=True),
    "max": df_raw.drop(columns=["date"]).max(numeric_only=True),
    "mean": df_raw.drop(columns=["date"]).mean(numeric_only=True),
    "std": df_raw.drop(columns=["date"]).std(numeric_only=True),
})
display(summary_table)

The dataset contains 3,201 daily observations from 2011 to 2023 with OHLCV for a single stock, plus benchmark, sector ETF, rates, implied vol, and news sentiment. One duplicate row. 12 missing values in news_sentiment_proxy — minor.

Key observation: open, high, low, close are non-stationary price levels. They must be transformed into returns or relative measures before modeling. Volume also has a suspicious floor at 500,000 in the summary stats — worth watching.

## 2. Define base features and the prediction target

In [ ]:
df = df_raw.sort_values("date").drop_duplicates().reset_index(drop=True).copy()

# Transform prices into returns and relative measures
for col in ['close', 'benchmark_close', 'sector_etf_close']:
    df[f"{col}_return"] = (df[col] / df[col].shift(1)) - 1

# Intraday range relative to close — captures daily volatility and microstructure
df['rel_open'] = (df['open'] / df['close'].shift(1)) - 1     # overnight gap
df['rel_low'] = (df['low'] / df['close']) - 1                # intraday drawdown
df['rel_high'] = (df['high'] / df['close']) - 1              # intraday rally

df = df.drop(['open', 'high', 'low', 'close', 'benchmark_close', 'sector_etf_close'], axis=1)

# Target: next-day return
df['target_return_next_day'] = df['close_return'].shift(-1)
df = df.iloc[1:-1].reset_index(drop=True).copy()

print("Shape after base transforms:", df.shape)
display(df[["date", "close_return", "target_return_next_day"]].head())

--> The OHLC transforms capture three distinct types of information: returns (trend), overnight gaps (rel_open), and intraday range (rel_low, rel_high). The target is the next-day close-to-close return, shifted by one day to prevent leakage.

## 3. Reserve a final holdout before detailed EDA

In [ ]:
holdout_frac = 0.20
holdout_size = int(len(df) * holdout_frac)

dev_df = df.iloc[:-holdout_size].copy()
holdout_df = df.iloc[-holdout_size:].copy()

display(pd.DataFrame({
    "sample": ["development", "final_holdout"],
    "rows": [len(dev_df), len(holdout_df)],
    "start_date": [dev_df["date"].min(), holdout_df["date"].min()],
    "end_date": [dev_df["date"].max(), holdout_df["date"].max()],
}))

--> The holdout covers Oct 2020 to Apr 2023, which includes the post-COVID recovery and the 2022 rate-hiking cycle. This is a materially different regime from the development period (2011–2020), so any model trained on the dev set faces a genuine out-of-sample challenge.

## 4. EDA on the development sample only

### 4.1 Stationarity

In [ ]:
# ADF test on daily returns (development sample)

from statsmodels.tsa.stattools import adfuller

ret = dev_df['close_return'].dropna()

adf_stat, adf_pvalue, _, _, adf_crit, _ = adfuller(ret, autolag="AIC")

adf_table = pd.DataFrame({
    "test": ["ADF"],
    "null_hypothesis": ["unit root / non-stationary"],
    "test_stat": [adf_stat],
    "p_value": [adf_pvalue],
})

display(adf_table)

print("ADF critical values:", adf_crit)

--> The ADF test strongly rejects the unit root hypothesis (p-value ≪ 1%), suggesting that daily returns can be treated as stationary in mean
--> Applying standard regression models on returns is theoretically justified from a stationarity perspective.

### 4.2 Return distribution

In [ ]:
# EDA on the distribution of daily returns (development sample)

from scipy.stats import skew, kurtosis

ret = dev_df["close_return"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
axes[0].hist(ret, bins=50, edgecolor="black", alpha=0.8)
axes[0].axvline(ret.mean(), linestyle="--", label=f"mean = {ret.mean():.4f}")
axes[0].axvline(ret.median(), linestyle=":", label=f"median = {ret.median():.4f}")
axes[0].set_title("Development sample: distribution of daily returns")
axes[0].set_xlabel("return_1d")
axes[0].set_ylabel("count")
axes[0].legend()

# Boxplot
axes[1].boxplot(ret, vert=True)
axes[1].set_title("Development sample: boxplot of daily returns")
axes[1].set_ylabel("return_1d")

# QQ plot
sm.qqplot(ret, line="s", ax=axes[2])
axes[2].set_title("Development sample: QQ plot of daily returns")

plt.tight_layout()
plt.show()

return_dist_summary = pd.DataFrame({
    "stat": [
        "count", "mean", "std", "min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max",
        "skewness", "excess_kurtosis"
    ],
    "value": [
        ret.count(),
        ret.mean(),
        ret.std(),
        ret.min(),
        ret.quantile(0.01),
        ret.quantile(0.05),
        ret.quantile(0.25),
        ret.quantile(0.50),
        ret.quantile(0.75),
        ret.quantile(0.95),
        ret.quantile(0.99),
        ret.max(),
        skew(ret, bias=False),
        kurtosis(ret, fisher=True, bias=False),
    ]
})

display(return_dist_summary.T)

--> Overall, the distribution is very close to Gaussian and exhibits some limited tail behavior, models relying on normality assumptions should work decently well. We notice mean return is slightly positive, pushing us to benchmark any start vs. buy and hold.

### 4.3 Outlier exploration

In [ ]:
# Outlier exploration on base features (before feature engineering)
base_features = ['volume', 'rates_change_bps', 'implied_vol_index', 'news_sentiment_proxy',
                 'close_return', 'benchmark_close_return', 'sector_etf_close_return',
                 'rel_open', 'rel_low', 'rel_high']

outlier_rows = []
for col in base_features:
    s = dev_df[col].dropna()
    if s.std() > 0:
        z_scores = (s - s.mean()) / s.std()
        outlier_pct = (z_scores.abs() > 3).mean() * 100
    else:
        outlier_pct = 0.0
    outlier_rows.append({"feature": col, "min": s.min(), "p01": s.quantile(0.01),
                         "p99": s.quantile(0.99), "max": s.max(),
                         "mean": s.mean(), "std": s.std(),
                         "abs_z_gt_3_pct": round(float(outlier_pct), 2)})
outlier_df = pd.DataFrame(outlier_rows).sort_values("abs_z_gt_3_pct", ascending=False)
display(outlier_df)

The outlier check looks broadly reassuring. Extreme values are moderately present, their frequency remains limited and the affected variables are precisely those where right tails are plausible.

I therefore do not see a strong case for removing observations at this stage. This looks more like natural skewness than a genuine data-quality issue.

### 4.4 Pairwise relationships and linear signal screen

In [ ]:
# pairwise analysis
base_features = ['volume', 'rates_change_bps', 'implied_vol_index', 'news_sentiment_proxy',
                 'benchmark_close_return', 'sector_etf_close_return',
                 'rel_open', 'rel_low', 'rel_high']

pair_sample = dev_df[base_features].dropna().sample(1000, random_state=42)
pp = sns.pairplot(pair_sample)
pp.fig.suptitle("Pairwise relationships on the development sample", y=1.02)
plt.show()

display(pair_sample.corr())

display(dev_df.drop('date', axis=1).dropna().corr()['target_return_next_day'].to_frame('corr_target'))

The correlation screen reveals four features with meaningful linear signal against the target:
- **close_return** (r = -0.14): short-term mean-reversion — today's up predicts tomorrow's down. This is one of the most documented anomalies in single-stock microstructure.
- **rel_high** (r = +0.12) and **rel_low** (r = +0.09): intraday range measures. Stocks that sold off hard intraday tend to bounce the next day — consistent with the mean-reversion story.
- **news_sentiment_proxy** (r = +0.09): positive news predicts positive next-day returns — consistent with slow information absorption.

Low multicollinearity between features, except between rel_low and rel_high (r = 0.32) which is mechanical.

### 4.5 Rolling correlation stability

In [ ]:
# Inspecting robustness of top 4 correlated featured through time

window = 125  # ~6 months of trading days

tmp = dev_df.copy()

inspected_features = ['close_return', 'news_sentiment_proxy', 'rel_low', 'rel_high']

news_expanding_mean = tmp["news_sentiment_proxy"].expanding().mean().shift(1)
tmp["news_sentiment_proxy"] = tmp["news_sentiment_proxy"].fillna(news_expanding_mean)

for feat in inspected_features:
    tmp[f"corr_{feat}"] = tmp[feat].rolling(window).corr(tmp["target_return_next_day"])
    tmp.plot(x='date', y=f"corr_{feat}", title=f"Rolling 6M correlation: {feat} vs target")

The rolling correlations fluctuate around their full-sample values but show no structural breaks or trend. close_return's negative correlation with the target is the most stable. news_sentiment oscillates more widely, suggesting it may be regime-dependent. Given more time, the next step would be to condition these correlations on the implied_vol_index to see if predictability concentrates in high-vol regimes.

In [ ]:
# Regime-conditional correlations: high vs low implied vol
vol_median = dev_df["implied_vol_index"].median()
dev_df["_high_vol"] = (dev_df["implied_vol_index"] > vol_median).astype(int)

top4 = ["close_return", "news_sentiment_proxy", "rel_low", "rel_high"]

regime_rows = []
for feat in top4:
    for regime, label in [(0, "low_vol"), (1, "high_vol")]:
        subset = dev_df.loc[dev_df["_high_vol"] == regime]
        corr = subset[feat].corr(subset["target_return_next_day"])
        regime_rows.append({"feature": feat, "regime": label, "n": len(subset), "corr_with_target": round(corr, 4)})

regime_corr_df = pd.DataFrame(regime_rows).pivot(index="feature", columns="regime", values="corr_with_target")
regime_corr_df["diff"] = regime_corr_df["high_vol"] - regime_corr_df["low_vol"]
display(regime_corr_df.sort_values("diff", ascending=False))

dev_df.drop(columns=["_high_vol"], inplace=True)

The regime analysis shows a split pattern: close_return and news_sentiment are more predictive in high-vol (mean-reversion weakens and news matters more when volatility is elevated), while rel_low and rel_high are more predictive in low-vol (intraday range signals work better in calm markets). The differences are modest (3-5 percentage points) — enough to justify exploring interaction features but not dramatic enough to warrant a fully regime-switching model. Given time constraints, I leave this as a next step.

### 4.6 Lag and momentum diagnostics

In [ ]:
# lag diag 
lag_diag = dev_df[["date", "target_return_next_day"]].copy()
for lag in [0, 1, 2, 5, 10, 240]:
    lag_diag[f"return_lag_{lag}"] = dev_df["close_return"].shift(lag)

fig, axes = plt.subplots(2, 3, figsize=(10, 8))
for ax, lag in zip(axes.ravel(), [0, 1, 2, 5, 10, 240]):
    sns.regplot(data=lag_diag, x=f"return_lag_{lag}", y="target_return_next_day", ax=ax,
                scatter_kws={"alpha":0.4, "s":18}, line_kws={"color":"black"})
    ax.set_title(f"target vs lag {lag}")
plt.tight_layout()
plt.show()

--> Clear mean-reversion signal at lag 0 (today's return vs tomorrow's) and a weaker signal at lag 1, motivating the creation of a yesterday_close_return feature.

In [ ]:
# momentum diag 

mom_diag = dev_df[["date", "target_return_next_day"]].copy()
for window in [5, 20, 30, 40, 120, 240]:
    mom_diag[f"momentum_{window}d_diag"] = dev_df["close_return"].rolling(window).mean()

fig, axes = plt.subplots(2, 3, figsize=(10, 8))
for ax, window in zip(axes.ravel(), [5, 20, 30, 40, 120, 240]):
    sns.regplot(data=mom_diag, x=f"momentum_{window}d_diag", y="target_return_next_day", ax=ax,
                scatter_kws={"alpha":0.4, "s":18}, line_kws={"color":"black"})
    ax.set_title(f"target vs momentum {window}d")
plt.tight_layout()
plt.show()

--> Visible signal at momentum windows 20/30/40/120/240. This is classic medium-term momentum — stocks that have been trending up tend to continue, net of the short-term mean-reversion. These windows are highly correlated with each other, so regularization or feature selection will be needed to avoid overfitting.

### 4.7 Feature significance with Newey-West standard errors

Before engineering new features, I formally test whether the base variables predict next-day returns. I use Newey-West (HAC) standard errors to account for potential autocorrelation in the residuals. I also split by implied volatility regime, as suggested by the conditional correlations in §4.5.

In [ ]:
sig_cols = [c for c in dev_df.select_dtypes(include=np.number).columns
            if c not in {"target_return_next_day"}]

sig_df = dev_df[["target_return_next_day"] + sig_cols].dropna()

formula = "target_return_next_day ~ " + " + ".join(sig_cols)
ols_full = smf.ols(formula=formula, data=sig_df).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("── Full development sample ──")
print(ols_full.summary())

In [ ]:
# ── Per-regime OLS ──
vol_median = dev_df["implied_vol_index"].median()
sig_df_regime = dev_df[["target_return_next_day"] + sig_cols].dropna()
sig_df_regime["_high_vol"] = (sig_df_regime["implied_vol_index"] > vol_median).astype(int)

for regime, label in [(1, "High implied vol"), (0, "Low implied vol")]:
    subset = sig_df_regime[sig_df_regime["_high_vol"] == regime]
    if len(subset) < 30:
        continue
    fit = smf.ols(formula=formula, data=subset).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"\n── {label} (n={len(subset)}) ──")
    coefs = pd.DataFrame({"coef": fit.params, "p_value": fit.pvalues}).drop("Intercept")
    display(coefs.round(4))

**Full sample** (n = 2,550, R² = 0.034, F-test p = 8e-15): the features are collectively significant. Individually, only two variables survive: **`news_sentiment_proxy`** (coef = 0.0019, p < 0.001) and **`implied_vol_index`** (coef = 0.0007, p = 0.027). All return-based features — including `close_return` (p = 0.84), `rel_high` (p = 0.69), `rel_low` (p = 0.80) — are non-significant despite their EDA correlations with the target.

The condition number is very large (1.58e9) and the covariance rank warning confirms severe **multicollinearity** among the return-based regressors (close, benchmark, sector, rel_open/low/high are all correlated daily returns). This explains why individually strong correlates (close_return at r = −0.14) lose significance in the multivariate OLS — their effects are absorbed by correlated variables.

**Per-regime split**:
- **High implied vol** (n = 1,274): `news_sentiment_proxy` is the only significant variable (p < 0.001). No return-based feature is significant.
- **Low implied vol** (n = 1,276): `news_sentiment_proxy` significant (p = 0.001), `implied_vol_index` borderline (p = 0.075).

The OLS confirms that `news_sentiment_proxy` carries the most robust linear signal. The return-based features (close_return, rel_high, rel_low) fail the significance test individually but may still contribute through Lasso-selected combinations or nonlinear interactions in tree-based models. The Lasso in §5 will resolve this by selecting the non-redundant subset.

## 5. Feature creation for modeling

All features are constructed causally using only past data (shift, rolling). I create momentum features at multiple scales and a lagged return, motivated by the EDA findings in sections 4.4–4.6.

In [ ]:
feat_df = df.copy()

# Clean news (causal: expanding mean imputation shifted by 1)
news_expanding_mean = feat_df["news_sentiment_proxy"].expanding().mean().shift(1)
feat_df["news_sentiment_proxy"] = feat_df["news_sentiment_proxy"].fillna(news_expanding_mean)

# Lag feature (motivated by lag diagnostic in 4.6)
feat_df["yesterday_close_return"] = feat_df["close_return"].shift(1)

# Momentum at multiple scales (motivated by momentum diagnostic in 4.6)
for window in [5, 20, 30, 40, 120, 240]:
    feat_df[f"momentum_{window}d"] = feat_df["close_return"].rolling(window).mean()

# Final cleanup
feat_df = feat_df.dropna().reset_index(drop=True)

print("Feature matrix shape:", feat_df.shape)

### Feature selection via Lasso (on development set only)

I use L1 regularization to identify which features carry genuine linear signal. The Lasso is fitted **only on the development portion** of feat_df to avoid leakage — the holdout must not influence feature selection.

In [ ]:
# Dynamic feature list: all numeric columns except date and target
excluded = {"date", "target_return_next_day"}
all_features = [c for c in feat_df.columns if c not in excluded]
all_features = feat_df[all_features].select_dtypes(include=np.number).columns.tolist()
print(f"Candidate features for Lasso: {len(all_features)}")

# Split feat_df into dev and holdout BEFORE Lasso
holdout_size_feat = int(len(feat_df) * holdout_frac)
dev_fe = feat_df.iloc[:-holdout_size_feat].copy()
holdout_fe = feat_df.iloc[-holdout_size_feat:].copy()

# Lasso on dev only
X_lasso = dev_fe[all_features]
y_lasso = dev_fe["target_return_next_day"]

lasso_pipe = Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(max_iter=10000, random_state=RANDOM_STATE))])
lasso_grid = GridSearchCV(lasso_pipe, {"lasso__alpha": np.logspace(-4, 1, 20)},
                          cv=TimeSeriesSplit(n_splits=5), scoring="neg_mean_squared_error", n_jobs=-1)
lasso_grid.fit(X_lasso, y_lasso)

coefs = lasso_grid.best_estimator_.named_steps["lasso"].coef_
selected = [f for f, c in zip(all_features, coefs) if abs(c) > 1e-8]

print("Best alpha:", lasso_grid.best_params_["lasso__alpha"])
print("Selected features:", selected)

lasso_coef_df = pd.DataFrame({"feature": all_features, "lasso_coef": coefs}).sort_values("lasso_coef", key=np.abs, ascending=False)
display(lasso_coef_df)

The Lasso selects 5 features: `close_return`, `yesterday_close_return`, `momentum_20d`, `implied_vol_index`, and `news_sentiment_proxy`. It zeroes out all other momentum scales, the OHLC relatives, volume, rates, and market/sector returns.

This is a very sparse selection, confirming that most of the linear signal concentrates in a handful of features — consistent with the severe multicollinearity flagged by the §4.7 OLS. The Lasso effectively resolves the collinearity by choosing one representative from each correlated group: `close_return` survives (despite OLS non-significance) because the L1 penalty picks it over the redundant return measures.

I keep the Lasso-selected features and add back market variables (benchmark, sector returns, rates, volume) that may carry nonlinear signal not visible to L1. Momentum scales not selected by the Lasso are dropped.

In [ ]:
# Final feature list: Lasso-selected + market variables that may carry nonlinear signal
features = [
    'close_return', 'yesterday_close_return',
    'momentum_20d', 'momentum_40d', 'momentum_120d',
    'volume', 'rates_change_bps', 'implied_vol_index',
    'news_sentiment_proxy', 'benchmark_close_return', 'sector_etf_close_return'
]

print("Final features:", len(features))

## 6. Development split for model selection

In [ ]:
test_frac = 0.20
test_size = int(len(dev_fe) * test_frac)
train_df = dev_fe.iloc[:-test_size].copy()
test_df = dev_fe.iloc[-test_size:].copy()

display(pd.DataFrame({
    "sample": ["train", "test", "holdout"],
    "rows": [len(train_df), len(test_df), len(holdout_fe)],
    "start_date": [train_df["date"].min(), test_df["date"].min(), holdout_fe["date"].min()],
    "end_date": [train_df["date"].max(), test_df["date"].max(), holdout_fe["date"].max()],
}))

X_train = train_df[features]
y_train = train_df["target_return_next_day"]
X_test = test_df[features]
y_test = test_df["target_return_next_day"]
X_holdout = holdout_fe[features]
y_holdout = holdout_fe["target_return_next_day"]

## 7. Models

In [ ]:
# ── Ridge ──
ridge_pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])
ridge_grid = {"model__alpha": np.logspace(-2, 2, 8)}
tscv = TimeSeriesSplit(n_splits=5)

grid_ridge = GridSearchCV(ridge_pipe, ridge_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=-1)
grid_ridge.fit(X_train, y_train)
ridge_best = grid_ridge.best_estimator_
ridge_pred_test = ridge_best.predict(X_test)

# ── Random Forest ──
rf_pipe = Pipeline([("model", RandomForestRegressor(random_state=RANDOM_STATE))])
rf_grid = {"model__n_estimators": [100, 300, 500], "model__max_depth": [2, 4, 6],
           "model__min_samples_leaf": [10, 30]}
grid_rf = GridSearchCV(rf_pipe, rf_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=-1)
grid_rf.fit(X_train, y_train)
rf_best = grid_rf.best_estimator_
rf_pred_test = rf_best.predict(X_test)

# ── Baselines ──
naive_pred_test = np.full(len(y_test), y_train.mean())
mom_benchmark_pred = test_df["momentum_20d"].values

# ── Results ──
results = []
for label, pred in [("Naive mean", naive_pred_test), ("Ridge", ridge_pred_test),
                     ("RF", rf_pred_test), ("Momentum 20d (raw)", mom_benchmark_pred)]:
    corr = float(np.corrcoef(y_test, pred)[0, 1]) if np.std(pred) > 1e-12 else np.nan
    results.append({"model": label, "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
                     "r2": float(r2_score(y_test, pred)), "corr": corr})

display(pd.DataFrame(results).sort_values("rmse"))
print("Best Ridge alpha:", grid_ridge.best_params_)
print("Best RF params:", grid_rf.best_params_)

Ridge and RF are marginally better than naive on RMSE (0.0165 vs 0.0167), with correlations of ~0.13. The strong regularization (alpha = 100) confirms low linear predictive power. The momentum_20d raw signal is essentially useless on its own (corr = −0.01), confirming that the ML models add real value by combining features — they are not just proxying a simple factor.

The RF selects very shallow trees (depth 2, leaf 10, 100 trees) — consistent with a weak, noisy signal where complexity hurts.

## 8. Model inspection

In [ ]:
ridge_coef = pd.DataFrame({
    "feature": features, "coef": ridge_best.named_steps["model"].coef_
}).sort_values("coef", ascending=False)
display(ridge_coef)

In [ ]:
rf_importance = pd.DataFrame({
    "feature": features, "importance": rf_best.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)
display(rf_importance)

Both models agree on the top predictors: `close_return` (mean-reversion), `news_sentiment_proxy` (information absorption), and `momentum_20d`/`momentum_40d` (trend continuation). The RF concentrates **54%** of importance on `close_return` — suggesting it uses today's return as a nonlinear conditioning variable, learning different mean-reversion strengths depending on the return magnitude. `news_sentiment_proxy` is second at 14%.

The Ridge coefficients tell a clean economic story: negative on `close_return` (−0.0026, mean-reversion), positive on `news_sentiment_proxy` (+0.0013, information absorption), positive on `momentum_20d` (+0.0009, trend continuation). The two effects coexist — short-term reversion plus medium-term momentum.

## 9. Ridge and RF diagnostics on the test set

In [ ]:
# Ridge 

ridge_diag = pd.DataFrame({
    "date": test_df["date"].values,
    "realized": y_test.values,
    "predicted": ridge_pred_test,
})
ridge_diag["residual"] = ridge_diag["realized"] - ridge_diag["predicted"]

plt.figure(figsize=(11, 4))
plt.plot(ridge_diag["date"], ridge_diag["realized"], label="realized")
plt.plot(ridge_diag["date"], ridge_diag["predicted"], label="ridge_pred")
plt.title("Ridge predictions vs realized returns on development test")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
sns.regplot(data=ridge_diag, x="predicted", y="realized", scatter_kws={"alpha":0.5, "s":20}, line_kws={"color":"black"})
plt.title("Ridge predicted vs realized returns")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(ridge_diag["date"], ridge_diag["residual"])
plt.title("Ridge residuals over time")
plt.tight_layout()
plt.show()

acf_vals = acf(ridge_diag["residual"], nlags=20, fft=False)
plt.figure(figsize=(8, 4))
plt.stem(range(len(acf_vals)), acf_vals)
plt.title("Autocorrelation of Ridge residuals")
plt.tight_layout()
plt.show()

display(acorr_ljungbox(ridge_diag["residual"], lags=[5, 10], return_df=True))


Few observations:
- Predicted returns are highly smoothed and remain close to zero compared to the much more volatile realized returns, consistently with the strong regularization factor
- The scatter plot shows a weak positive relationship between predictions and realized values, suggesting some minor predictive power
- Residuals are centered around zero with no visible trend over time, indicating no systematic bias in the model
- The autocorrelation function (ACF) of residuals shows no significant lags beyond zero, suggesting absence remaining of linear temporal dependence
- Ljung-Box test p-values are high, confirming that residuals are consistent with white noise (no autocorrelation)

--> Overall, the model is statistically well-behaved (uncorrelated residuals) but captures little signal, making its predictive performance weak.

In [ ]:
# RF

rf_diag = pd.DataFrame({
    "date": test_df["date"].values,
    "realized": y_test.values,
    "predicted": rf_pred_test,
})
rf_diag["residual"] = rf_diag["realized"] - rf_diag["predicted"]

# --- Time series plot ---
plt.figure(figsize=(11, 4))
plt.plot(rf_diag["date"], rf_diag["realized"], label="realized")
plt.plot(rf_diag["date"], rf_diag["predicted"], label="rf_pred")
plt.title("Random Forest predictions vs realized returns on development test")
plt.legend()
plt.tight_layout()
plt.show()

# --- Scatter plot ---
plt.figure(figsize=(6, 5))
sns.regplot(
    data=rf_diag,
    x="predicted",
    y="realized",
    scatter_kws={"alpha": 0.5, "s": 20},
    line_kws={"color": "black"}
)
plt.title("Random Forest predicted vs realized returns")
plt.tight_layout()
plt.show()

# --- Residuals over time ---
plt.figure(figsize=(11, 3.5))
plt.plot(rf_diag["date"], rf_diag["residual"])
plt.title("Random Forest residuals over time")
plt.tight_layout()
plt.show()

# --- ACF of residuals ---
acf_vals = acf(rf_diag["residual"], nlags=20, fft=False)
plt.figure(figsize=(8, 4))
plt.stem(range(len(acf_vals)), acf_vals)
plt.title("Autocorrelation of RF residuals")
plt.tight_layout()
plt.show()

# --- Ljung-Box test ---
display(acorr_ljungbox(rf_diag["residual"], lags=[5, 10], return_df=True))

Same conclusion as Ridge: well-behaved residuals (no autocorrelation, centered around zero), but the model captures little of the total variance. The signal is real but small.

## 10. Walk-forward validation — signal stability over time

Before translating predictions into a trading strategy, I verify that the Ridge signal is stable across time. The walk-forward uses `TimeSeriesSplit` on the full development set: each fold trains on expanding historical data and tests on a subsequent block. A consistently positive predictive correlation across folds indicates temporal robustness.

In [ ]:
wf_df = dev_fe.copy().dropna(subset=features + ["target_return_next_day"]).reset_index(drop=True)

X_wf = wf_df[features]
y_wf = wf_df["target_return_next_day"]
dates_wf = wf_df["date"]

tscv_wf = TimeSeriesSplit(n_splits=5)

wf_rows = []
for fold, (train_idx, test_idx) in enumerate(tscv_wf.split(X_wf), start=1):
    X_tr, y_tr = X_wf.iloc[train_idx], y_wf.iloc[train_idx]
    X_te, y_te = X_wf.iloc[test_idx], y_wf.iloc[test_idx]
    d_te = dates_wf.iloc[test_idx]

    ridge_fold = Pipeline([("scaler", StandardScaler()),
        ("model", Ridge(alpha=grid_ridge.best_params_["model__alpha"]))])
    ridge_fold.fit(X_tr, y_tr)
    y_pred = ridge_fold.predict(X_te)

    fold_corr = np.corrcoef(y_te, y_pred)[0, 1] if np.std(y_pred) > 1e-12 else np.nan

    wf_rows.append({
        "fold": fold,
        "train_end": dates_wf.iloc[train_idx].max().strftime("%Y-%m-%d"),
        "test_start": d_te.min().strftime("%Y-%m-%d"),
        "test_end": d_te.max().strftime("%Y-%m-%d"),
        "n_train": len(train_idx), "n_test": len(test_idx),
        "rmse": round(np.sqrt(mean_squared_error(y_te, y_pred)), 6),
        "corr": round(fold_corr, 4),
    })

wf_results = pd.DataFrame(wf_rows)
display(wf_results)
print(f"\nMean correlation: {wf_results['corr'].mean():.4f} ± {wf_results['corr'].std():.4f}")

The Ridge walk-forward shows consistently positive predictive correlation across all 5 folds, ranging from 0.128 to 0.209 with a mean of **0.163 ± 0.030**. No fold is negative. This temporal stability is a necessary condition before translating the signal into a trading strategy — a high average correlation driven by a single fold would be much less trustworthy.

The magnitude (~0.16) is notably high for a daily single-stock prediction task. While the Lasso feature selection is done strictly on the dev set (avoiding leakage), the walk-forward correlation warrants a careful audit that no forward-looking information leaked into the base feature construction.

## 11. Translate predictions into trading strategies

In [ ]:
def backtest_from_prediction(y_true, y_pred, dates, cost_per_turnover=0.0005, clip_at=1.0):
    y_true = pd.Series(np.asarray(y_true), index=pd.Index(dates), name="asset_return")
    y_pred = pd.Series(np.asarray(y_pred), index=pd.Index(dates), name="prediction")

    scale = y_pred.std() + 1e-12
    position = (y_pred / scale).clip(-clip_at, clip_at)

    turnover = position.diff().abs().fillna(position.abs())
    gross = position * y_true
    net = gross - cost_per_turnover * turnover

    out = pd.DataFrame({
        "date": y_true.index,
        "asset_return": y_true.values,
        "prediction": y_pred.values,
        "position": position.values,
        "turnover": turnover.values,
        "strategy_return": net.values,
    }).set_index("date")

    out["cum_return"] = (1 + out["strategy_return"]).cumprod()
    return out


def backtest_hold(y_true, dates, cost_per_turnover=0.0005):
    y_true = pd.Series(np.asarray(y_true), index=pd.Index(dates), name="asset_return")
    position = pd.Series(1.0, index=y_true.index, name="position")

    turnover = position.diff().abs().fillna(position.abs())
    gross = position * y_true
    net = gross - cost_per_turnover * turnover

    out = pd.DataFrame({
        "date": y_true.index,
        "asset_return": y_true.values,
        "prediction": np.nan,
        "position": position.values,
        "turnover": turnover.values,
        "strategy_return": net.values,
    }).set_index("date")

    out["cum_return"] = (1 + out["strategy_return"]).cumprod()
    return out


# Refit final models on all development data
X_dev_all = dev_fe[features]
y_dev_all = dev_fe["target_return_next_day"]

final_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=grid_ridge.best_params_["model__alpha"]))
])
final_ridge.fit(X_dev_all, y_dev_all)

final_rf = RandomForestRegressor(
    random_state=42,
    n_estimators=grid_rf.best_params_["model__n_estimators"],
    max_depth=grid_rf.best_params_["model__max_depth"],
    min_samples_leaf=grid_rf.best_params_["model__min_samples_leaf"],
)
final_rf.fit(X_dev_all, y_dev_all)

# Predictions on full sample
feat_df = feat_df.copy()
feat_df["ridge_pred_fullsample_refit"] = final_ridge.predict(feat_df[features])
feat_df["rf_pred_fullsample_refit"] = final_rf.predict(feat_df[features])

# Predictions on holdout
holdout_ridge_pred = final_ridge.predict(X_holdout)
holdout_rf_pred = final_rf.predict(X_holdout)

# Full sample backtests - strength only
full_ridge_strength_bt = backtest_from_prediction(
    feat_df["target_return_next_day"],
    feat_df["ridge_pred_fullsample_refit"],
    feat_df["date"]
)

full_rf_strength_bt = backtest_from_prediction(
    feat_df["target_return_next_day"],
    feat_df["rf_pred_fullsample_refit"],
    feat_df["date"]
)

full_hold_bt = backtest_hold(
    feat_df["target_return_next_day"],
    feat_df["date"]
)

# Holdout backtests - strength only
holdout_ridge_strength_bt = backtest_from_prediction(
    y_holdout,
    holdout_ridge_pred,
    holdout_fe["date"]
)

holdout_rf_strength_bt = backtest_from_prediction(
    y_holdout,
    holdout_rf_pred,
    holdout_fe["date"]
)

holdout_hold_bt = backtest_hold(
    y_holdout,
    holdout_fe["date"]
)

In [ ]:
def strategy_summary(bt_df, label):
    r = bt_df["strategy_return"]
    mean_daily = r.mean()
    std_daily = r.std()
    sharpe = np.sqrt(252) * mean_daily / (std_daily + 1e-12)
    ann_return = (1 + mean_daily) ** 252 - 1
    max_dd = (bt_df["cum_return"] / bt_df["cum_return"].cummax() - 1).min()

    return {
        "strategy": label,
        "mean_daily_return": float(mean_daily),
        "annualized_return": float(ann_return),
        "annualized_vol": float(std_daily * np.sqrt(252)),
        "sharpe": float(sharpe),
        "max_drawdown": float(max_dd),
        "avg_turnover": float(bt_df["turnover"].mean()),
        "annualized_turnover": float(bt_df["turnover"].mean() * 252),
    }

full_perf = pd.DataFrame([
    strategy_summary(full_ridge_strength_bt, "Ridge strength"),
    strategy_summary(full_rf_strength_bt, "RF strength"),
    strategy_summary(full_hold_bt, "Hold"),
]).sort_values("sharpe", ascending=False)

holdout_perf = pd.DataFrame([
    strategy_summary(holdout_ridge_strength_bt, "Ridge strength"),
    strategy_summary(holdout_rf_strength_bt, "RF strength"),
    strategy_summary(holdout_hold_bt, "Hold"),
]).sort_values("sharpe", ascending=False)

print("Full-sample trading performance")
display(full_perf)

print("Final holdout trading performance")
display(holdout_perf)

In [ ]:
# --- Full sample cumulative returns ---
plt.figure(figsize=(11, 4))
plt.plot(full_ridge_strength_bt.index, full_ridge_strength_bt["cum_return"], label="Ridge strength")
plt.plot(full_rf_strength_bt.index, full_rf_strength_bt["cum_return"], label="RF strength")
plt.plot(full_hold_bt.index, full_hold_bt["cum_return"], label="Hold")
plt.title("Trading strategies: full sample")
plt.legend()
plt.tight_layout()
plt.show()

# --- Holdout cumulative returns ---
plt.figure(figsize=(11, 4))
plt.plot(holdout_ridge_strength_bt.index, holdout_ridge_strength_bt["cum_return"], label="Ridge strength")
plt.plot(holdout_rf_strength_bt.index, holdout_rf_strength_bt["cum_return"], label="RF strength")
plt.plot(holdout_hold_bt.index, holdout_hold_bt["cum_return"], label="Hold")
plt.title("Trading strategies: final holdout only")
plt.legend()
plt.tight_layout()
plt.show()

# --- Turnover (full sample) ---
plt.figure(figsize=(10, 4))
plt.plot(full_ridge_strength_bt.index, full_ridge_strength_bt["turnover"].rolling(20).mean(), label="Ridge strength")
plt.plot(full_rf_strength_bt.index, full_rf_strength_bt["turnover"].rolling(20).mean(), label="RF strength")
plt.title("20-day rolling turnover: model-based strategies (full sample)")
plt.legend()
plt.tight_layout()
plt.show()

Both Ridge and RF deliver strong holdout Sharpes: **2.44** and **2.41** respectively, materially above buy-and-hold (1.34). The Sharpe decay from full-sample to holdout is modest (Ridge: 2.39 → 2.44, RF: 2.80 → 2.41), suggesting the signal generalizes well to the post-COVID holdout period. Ridge actually *improves* slightly on the holdout, while RF shows a modest decay consistent with mild overfitting.

The model-based strategies beat buy-and-hold on the holdout with roughly 75% of the volatility (19% vs 26% annualized). This risk-adjusted outperformance is the economically meaningful result — the models are not just timing the market but doing so with lower drawdown (−15% to −17% vs −22% for hold).

In [ ]:
# Sensitivity of holdout RF-strength Sharpe to transaction costs

cost_bps_grid = [0, 5, 10, 15, 20, 25, 30]

rf_cost_sensitivity_rows = []

for cost_bps in cost_bps_grid:
    cost_decimal = cost_bps / 10000.0  # 1 bp = 0.0001
    
    bt = backtest_from_prediction(
        y_holdout,
        holdout_rf_pred,
        holdout_fe["date"],
        cost_per_turnover=cost_decimal
    )
    
    perf = strategy_summary(bt, f"Rf strength ({cost_bps} bps)")
    
    rf_cost_sensitivity_rows.append({
        "transaction_cost_bps": cost_bps,
        "mean_daily_return": perf["mean_daily_return"],
        "annualized_return": perf["annualized_return"],
        "annualized_vol": perf["annualized_vol"],
        "sharpe": perf["sharpe"],
        "max_drawdown": perf["max_drawdown"],
        "avg_turnover": perf["avg_turnover"],
        "annualized_turnover": perf["annualized_turnover"],
    })

rf_cost_sensitivity = pd.DataFrame(rf_cost_sensitivity_rows)

print("Holdout sensitivity of Rf-strength strategy to transaction costs")
display(rf_cost_sensitivity)

The RF strategy remains profitable up to ~25 bps (Sharpe ≈ 0.19), with breakeven around 27 bps. At realistic institutional costs of 5–10 bps for a liquid single stock, the Sharpe stays above 1.9–2.4 — a comfortable margin. However, the annualized turnover is high (~210 for RF, ~195 for Ridge), meaning the strategy trades aggressively and is sensitive to execution quality.

The sensitivity is roughly linear: each 5 bps of cost reduces the Sharpe by ~0.55. This linearity confirms that the alpha is broadly distributed across trading days rather than concentrated in a few large bets.

## 12. Statistical inference on strategy returns

In [ ]:
def iid_ttest_summary(strategy_returns):
    r = pd.Series(strategy_returns).dropna()
    t_stat, p_value = stats.ttest_1samp(r, 0.0)
    return {
        "n_obs": int(len(r)),
        "mean_return": float(r.mean()),
        "t_stat_iid": float(t_stat),
        "p_value_iid": float(p_value),
    }


def newey_west_mean_test(strategy_returns, maxlags=5):
    r = pd.Series(strategy_returns).dropna()
    X = np.ones((len(r), 1))
    fit = sm.OLS(r.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return {
        "mean_return": float(fit.params[0]),
        "se_newey_west": float(fit.bse[0]),
        "t_stat_newey_west": float(fit.tvalues[0]),
        "p_value_newey_west": float(fit.pvalues[0]),
        "maxlags": int(maxlags),
    }

In [ ]:
# Holdout inference: use one consistent naming convention everywhere
holdout_inference = {
    "Ridge strength": holdout_ridge_strength_bt["strategy_return"],
    "Rf strength": holdout_rf_strength_bt["strategy_return"],
}

iid_rows = []
nw_rows = []
raw_pvals = []

for label, r in holdout_inference.items():
    iid_stats = iid_ttest_summary(r)
    iid_rows.append({
        "strategy": label,
        "n_obs": iid_stats["n_obs"],
        "mean_return": iid_stats["mean_return"],
        "t_stat_iid": iid_stats["t_stat_iid"],
        "p_value_iid": iid_stats["p_value_iid"],
    })

    nw_stats = newey_west_mean_test(r, maxlags=5)
    nw_rows.append({
        "strategy": label,
        "mean_return": nw_stats["mean_return"],
        "se_newey_west": nw_stats["se_newey_west"],
        "t_stat_newey_west": nw_stats["t_stat_newey_west"],
        "p_value_newey_west": nw_stats["p_value_newey_west"],
        "maxlags": nw_stats["maxlags"],
    })

    raw_pvals.append(nw_stats["p_value_newey_west"])

iid_table = pd.DataFrame(iid_rows)
nw_table = pd.DataFrame(nw_rows)

reject, pvals_bonf, _, _ = multipletests(raw_pvals, alpha=0.05, method="bonferroni")
nw_table["p_value_bonferroni"] = pvals_bonf
nw_table["reject_5pct_after_bonferroni"] = reject

print("IID t-test summary (holdout only)")
display(iid_table)

print("Newey-West mean test with Bonferroni correction across tested strategies (holdout only)")
display(nw_table)

Both strategies are statistically significant after Bonferroni correction: Ridge (NW t = 3.69, p = 0.0002, Bonferroni p = 0.0004) and RF (NW t = 3.48, p = 0.0005, Bonferroni p = 0.001). With 591 holdout observations and HAC-corrected standard errors, the alpha is real by conventional statistical standards.

The t-statistics under Newey-West are well above 2.0, and the Bonferroni correction — which is conservative — does not change the conclusion. This is a strong result that is consistent with the walk-forward stability (§10) and the modest Sharpe decay from in-sample to holdout (§11).

## 13. Robustness: regime decomposition

To check whether the alpha is regime-dependent, I split the holdout Ridge strategy returns by implied volatility regime and run Newey-West tests on each subset. This parallels the EDA finding (§4.5, §4.7) that predictability may concentrate in high-vol periods.

In [ ]:
r_ridge_all = holdout_ridge_strength_bt["strategy_return"].copy()

regime_df = holdout_fe[["date", "implied_vol_index"]].copy()
vol_median_holdout = holdout_fe["implied_vol_index"].median()
regime_df["_high_vol"] = (regime_df["implied_vol_index"] > vol_median_holdout).astype(int)
regime_df = regime_df.merge(r_ridge_all.rename("strategy_return"),
                             left_on="date", right_index=True, how="inner")

r_ridge_high = regime_df.loc[regime_df["_high_vol"] == 1, "strategy_return"]
r_ridge_low = regime_df.loc[regime_df["_high_vol"] == 0, "strategy_return"]

robustness_rows = []
for label, r in [("Ridge (all)", r_ridge_all), ("Ridge high vol", r_ridge_high), ("Ridge low vol", r_ridge_low)]:
    iid = iid_ttest_summary(r)
    nw = newey_west_mean_test(r, maxlags=5)
    robustness_rows.append({"strategy": label, "n_obs": iid["n_obs"], "mean_return": iid["mean_return"],
        "t_stat_nw": nw["t_stat_newey_west"], "p_value_nw": nw["p_value_newey_west"]})

display(pd.DataFrame(robustness_rows).sort_values("t_stat_nw", ascending=False))

The regime decomposition reveals that the signal is present in **both** regimes, but with an interesting asymmetry:
- **Low implied vol** (n = 296): mean daily return 0.22%, NW t = 3.35, **p = 0.0008** — highly significant
- **High implied vol** (n = 295): mean daily return 0.15%, NW t = 2.21, **p = 0.027** — significant but weaker

The alpha is stronger in calm markets. This is economically consistent with a signal dominated by short-term mean-reversion: in low-vol environments, prices revert to trend more reliably. In high-vol, mean-reversion is noisier — sharp moves can persist or reverse unpredictably, diluting the signal.

This regime pattern is reassuring: the strategy does not depend on crisis periods to generate returns. It works in normal markets and partially survives stress — making it more robust as a production signal than one that concentrates exclusively in extreme regimes.

## Final remarks

The analysis identifies a statistically significant and economically meaningful predictive signal for single-stock daily returns. The core drivers — short-term mean-reversion (`close_return`), momentum at 20–40 day horizons, and news sentiment absorption — are economically interpretable and temporally stable across 5 walk-forward folds (mean correlation 0.163, std 0.030).

**Key findings:**

**The signal is real and survives rigorous testing.** Both Ridge and RF beat buy-and-hold on the holdout (Sharpe ~2.4 vs 1.3), and both pass Bonferroni-corrected Newey-West tests at the 0.1% level. The walk-forward validation shows no fold with negative correlation. The Lasso feature selection is done exclusively on the dev set, avoiding leakage.

**The signal works across volatility regimes.** The robustness analysis (§13) shows significance in both low-vol (p = 0.0008) and high-vol (p = 0.027), with stronger performance in calm markets — consistent with mean-reversion being more reliable when volatility is low.

**Transaction costs are manageable but non-trivial.** At 5 bps, the holdout Sharpe drops from ~2.96 to ~2.41 — a significant drag but still well above the buy-and-hold benchmark. The breakeven is at ~27 bps, providing a meaningful buffer for a liquid single stock.

**The model captures two coexisting effects.** The Ridge coefficients encode both mean-reversion (negative on `close_return`) and momentum (positive on `momentum_20d`). The RF concentrates 54% of importance on `close_return`, suggesting it learns a nonlinear mean-reversion function — likely conditioning the reversion strength on the magnitude of today's return.

**Caveats:**
- The walk-forward correlation of 0.16 is high for a pure time-series setting and warrants a thorough audit that no forward-looking information leaked into the features
- The holdout covers a specific macro regime (post-COVID, rate hiking) — broader validation across regimes would strengthen the conclusion
- The Lasso selects only 5 features, suggesting the signal is narrow; this makes the strategy concentrated on a few patterns that could decay

**What I would do next:**
- **Audit feature construction** carefully — the 0.16 walk-forward correlation is unusually high and demands scrutiny
- **Reduce turnover**: apply a position filter or move to weekly rebalancing to lower cost sensitivity
- **Extend the walk-forward to the full sample** (including holdout) to test across more regimes
- **Test on additional stocks or a small universe** to check whether the signal generalizes